# ComfyUI + PromptDB
通用風景範例及私人 Drive 匯入。選 GPU runtime 後依序執行。
不包含本地私人提示、工作流或模型。請先閱讀 [整合說明](https://github.com/JackChen6203/Google-Colab_Notebooks/blob/codex/colab-promptdb/promptdb_colab/README.md)。

In [ ]:
from pathlib import Path
import subprocess, sys, json
ROOT = Path('/content')
REPO = ROOT / 'Google-Colab_Notebooks'
COMFY = ROOT / 'ComfyUI'
def run(*args):
    subprocess.run([str(a) for a in args], check=True)
if not REPO.exists():
    run('git', 'clone', '--branch', 'codex/colab-promptdb', '--single-branch', 'https://github.com/JackChen6203/Google-Colab_Notebooks.git', REPO)
if not COMFY.exists():
    run('git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', COMFY)
    run('git', '-C', COMFY, 'checkout', '0b05cd8a3683cbf0c291b1472bb8b64da41de9ae')
run(sys.executable, '-m', 'pip', 'install', '-r', COMFY / 'requirements.txt')
run('nvidia-smi')


## 選用：私人 Drive
掛載後 runtime 可存取 Drive。預設只使用附帶的風景範例。

In [ ]:
USE_DRIVE = False # @param {type:"boolean"}
DATA = None
WORKFLOWS = None
MANIFEST = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PRIVATE = Path('/content/drive/MyDrive/ComfyUI-PromptDB')
    DATA = PRIVATE / 'user-db'
    WORKFLOWS = PRIVATE / 'workflows'
    MANIFEST = PRIVATE / 'models.json'
    if not DATA.is_dir():
        raise FileNotFoundError(f'Create {DATA} and add general-purpose prompt JSON files first')


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('promptdb_setup', REPO / 'promptdb_colab/setup.py')
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)
helper.install(COMFY, DATA)
if MANIFEST and MANIFEST.is_file():
    helper.download_models(COMFY, MANIFEST)
print('PromptDB installed. Start ComfyUI next.')


## 啟動 ComfyUI
此儲存格等待最多 60 秒。若失敗，查看 `/content/comfyui.log`。

In [ ]:
import time, urllib.request
log = open('/content/comfyui.log', 'a')
if 'process' not in globals() or process.poll() is not None:
    process = subprocess.Popen([sys.executable, str(COMFY / 'main.py'), '--listen', '127.0.0.1', '--port', '8188'], cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)
for attempt in range(60):
    if process.poll() is not None:
        raise RuntimeError(Path('/content/comfyui.log').read_text()[-6000:])
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/object_info', timeout=2) as response:
            info = json.load(response)
        if 'PromptDB' not in info:
            raise RuntimeError('PromptDB failed to load; inspect /content/comfyui.log')
        break
    except (OSError, TimeoutError):
        time.sleep(1)
else:
    raise TimeoutError('ComfyUI not ready; inspect /content/comfyui.log')
from google.colab.output import eval_js
from IPython.display import display, HTML
url = eval_js('google.colab.kernel.proxyPort(8188)')
import html
display(HTML(f'<a href="{html.escape(url, quote=True)}" target="_blank">Open ComfyUI</a>'))


## 檢查載入的 PromptDB 與工作流依賴
這是文字/API 與檔名檢查；出圖、模型相容性及完整子圖依賴仍須實測。

In [ ]:
request = urllib.request.Request('http://127.0.0.1:8188/prompt_db_categories', data=json.dumps({'database':'krea2'}).encode(), headers={'Content-Type':'application/json'})
with urllib.request.urlopen(request) as response:
    print('PromptDB categories:', response.read().decode())
if WORKFLOWS and WORKFLOWS.is_dir():
    report = helper.audit(COMFY, WORKFLOWS, 'http://127.0.0.1:8188')
    Path('/content/workflow_dependencies.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('No private workflows selected.')
